# » `Dependencias`:

In [ ]:
# !pip install plotly
# !pip install sqlalchemy
# !pip install numpy
# !pip install matplotlib
# !pip install PIL
# !pip install io

In [3]:
import pandas as pd
from sqlalchemy import create_engine
import os
from PIL import Image
from io import BytesIO

## 1. Conección → SIM(172.27.0.124)

In [4]:
# spring.datasource.url = jdbc:sqlserver://172.27.250.27;databaseName=SIRIM
SERVER = '172.27.0.124' # '172.27.0.242'
#DRIVER = 'SQL Server Native Client 11.0'
DRIVER = 'ODBC Driver 17 for SQL Server'
DATABASE = 'SIM'
USERNAME = 'userestadistica' # 'udesa'
PASSWORD = '$Us3R_3sT4d1sTic4$' # 'DESARROLLO2006'
DATABASE_CONNECTION = f'mssql://{USERNAME}:{PASSWORD}@{SERVER}/{DATABASE}?driver={DRIVER}'

engine = create_engine(DATABASE_CONNECTION)
connection = engine.connect()

## 2. Métodos genérico:

In [5]:
def get_query_sql(query):
  try:
    df = pd.read_sql(query, connection)
    return df
  except:
    print('¡Ocurrió un error!')

## 3. Lectura de ...

### 3.1 ...

In [21]:
# Extracción
SQL_QUERY = 'SELECT * FROM SimImagenVen411'

df_imgs = get_query_sql(SQL_QUERY)

In [27]:
#['sFileName', 'sImageName', 'xImagen']

df_file = df_imgs[['sFileName']].drop_duplicates().reset_index(drop=True)

In [37]:
# Crear la carpeta si no existe
carpeta = 'imagenes'
if not os.path.exists(carpeta):
    os.makedirs(carpeta)

# Convertir el campo VARBINARY a imagen y depositarlas en la carpeta
for i, file in df_file.iterrows():
    file_name = file['sFileName']
    path_root = os.path.join('img', file_name)

    # Crear la carpeta
    os.makedirs(path_root)

    # imagenes
    df_imgs_of_curr_file = df_imgs.loc[df_imgs['sFileName'] == file_name]

    for ii, img in df_imgs_of_curr_file.iterrows():
      img_name = img['sImageName']
      img_bytes = img['xImagen']
      imagen = Image.open(BytesIO(img_bytes))
      imagen_rgb = imagen.convert('RGB')
      imagen_rgb.save(os.path.join(path_root, img_name))